In [6]:
import os
import zipfile
import re
import math
import pandas as pd
from collections import defaultdict, Counter
import gc

print("--- STARTING PIPELINE: BROWN CORPUS (SKETCH ENGINE ALDF LOGIC) ---")

# ==========================================
# SETUP: FETCH CLEAN BROWN CORPUS FROM NLTK
# ==========================================
if not os.path.exists("brown_corpus"):
    print("Fetching and setting up Brown corpus via NLTK...")
    import nltk
    nltk.download('brown', quiet=True)
    from nltk.corpus import brown
    os.makedirs("brown_corpus", exist_ok=True)
    for fileid in brown.fileids():
        raw_text = " ".join(brown.words(fileid))
        with open(os.path.join("brown_corpus", f"{fileid}.txt"), 'w', encoding='utf-8') as out_f:
            out_f.write(raw_text)
    print("Brown corpus files written to 'brown_corpus/' folder successfully.")

# ==========================================
# FORCE UNZIP: LOOK FOR ANY ZIP FILE UPLOADED
# ==========================================
if not os.path.exists("my_essays"):
    found_zip = False
    for file in os.listdir():
        if file.endswith(".zip") and "archive" not in file.lower():
            print(f"Found zip file: {file}. Unzipping to 'my_essays'...")
            with zipfile.ZipFile(file, 'r') as zip_ref:
                zip_ref.extractall("my_essays")
            found_zip = True
            break
    if not found_zip:
        print("CRITICAL: No zip file found in the sidebar. Please upload your zip now.")

# ==========================================
# TOKENIZER (NO LEMMATIZATION)
# ==========================================
pattern = re.compile(r"[^\W\d_]+(?:-[^\W\d_]+)*", re.UNICODE)

# ==========================================
# PHASE 1: BUILD MASTER DICTIONARY
# ==========================================
print("\n--- PHASE 1: BUILDING MASTER DICTIONARY ---")
word_positions = defaultdict(list)
global_position = 0

for root, dirs, files in os.walk("brown_corpus"):
    for file in sorted(files):
        if file.endswith(".txt"):
            filepath = os.path.join(root, file)
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                tokens = pattern.findall(f.read().lower())
                for token in tokens:
                    word_positions[token].append(global_position)
                    global_position += 1

TOTAL_TOKENS = global_position
print(f"Total tokens processed: {TOTAL_TOKENS:,}")

gc.collect()

print("Calculating ALDF (Sketch Engine circular-distance logic)...")
ref_dict = {}
dict_results = []

for token, positions in word_positions.items():
    freq = len(positions)
    if freq >= 5:
        d_list = []
        for i in range(1, freq):
            d_list.append(positions[i] - positions[i - 1])
        d_list.append((positions[0] + TOTAL_TOKENS) - positions[-1])
        ald = sum((d / TOTAL_TOKENS) * math.log2(d) for d in d_list if d > 0)
        aldf_score = round(TOTAL_TOKENS / (2 ** ald), 4)
        ref_dict[token] = aldf_score
        dict_results.append({'Token': token, 'Freq': freq, 'ALDF': aldf_score})

pd.DataFrame(dict_results).sort_values(by='Freq', ascending=False).to_csv("ALDF_Master_Dictionary.csv", index=False)
print("Dictionary saved as 'ALDF_Master_Dictionary.csv'.")
del word_positions; del dict_results; gc.collect()

# ==========================================
# PHASE 2: SCORE STUDENT ESSAYS
# ==========================================
print("\n--- PHASE 2: SCORING WORKSHOP ESSAYS ---")

function_words = set([
    "the", "a", "an", "some", "any", "every", "all", "both", "neither", "either", "no", "each", "another",
    "i", "me", "my", "mine", "we", "us", "our", "ours", "you", "your", "yours",
    "he", "him", "his", "she", "her", "hers", "it", "its", "they", "them", "their", "theirs",
    "this", "that", "these", "those", "in", "on", "at", "by", "for", "with", "about", "against", "between", "into", "through",
    "during", "before", "after", "above", "below", "to", "from", "up", "down", "of", "over", "under", "upon", "within", "without", "among", "beyond", "behind",
    "and", "but", "or", "yet", "so", "nor", "although", "because", "since", "unless", "while", "if", "than", "as", "until",
    "is", "am", "are", "was", "were", "be", "being", "been", "has", "have", "had", "do", "does", "did",
    "will", "would", "shall", "should", "can", "could", "may", "might", "must", "here", "there", "when", "where", "why", "how", "then", "once", "again", "further", "too", "very", "not"
])

results = []
essays_dir = "my_essays"

if os.path.exists(essays_dir):
    essay_count = 0
    for root, dirs, files in os.walk(essays_dir):
        for filename in files:
            if filename.endswith(".txt"):
                essay_count += 1
                filepath = os.path.join(root, filename)
                with open(filepath, 'r', encoding='utf-8', errors='ignore') as file:
                    text = file.read().lower()
                    tokens = pattern.findall(text)
                    token_counts = Counter(tokens)

                    all_total, all_count = 0, 0
                    func_total, func_count = 0, 0
                    content_total, content_count = 0, 0

                    for token, freq in token_counts.items():
                        if token in ref_dict:
                            score = ref_dict[token]
                            all_total += freq * score
                            all_count += freq
                            if token in function_words:
                                func_total += freq * score
                                func_count += freq
                            else:
                                content_total += freq * score
                                content_count += freq

                    mean_all = all_total / all_count if all_count else 0
                    mean_func = func_total / func_count if func_count else 0
                    mean_content = content_total / content_count if content_count else 0
                    coverage = (all_count / len(tokens)) * 100 if tokens else 0

                    results.append({
                        "Essay": filename,
                        "Total_Tokens": len(tokens),
                        "Coverage_Percent": round(coverage, 2),
                        "ALDF_AW": round(mean_all, 4),
                        "ALDF_Function": round(mean_func, 4),
                        "ALDF_Content": round(mean_content, 4)
                    })

    if results:
        pd.DataFrame(results).to_csv("ALDF_Scores.csv", index=False)
        print(f"SUCCESS! Processed {essay_count} essays.")
        print("Results saved to 'ALDF_Scores.csv'.")
    else:
        print("Found the zip, but no .txt files were inside it.")
else:
    print("Phase 2 skipped because no zip file was found to unzip.")

--- STARTING PIPELINE: BROWN CORPUS (SKETCH ENGINE ALDF LOGIC) ---
Found zip file: Essays_for_ISTAL27_Workshop.zip. Unzipping to 'my_essays'...

--- PHASE 1: BUILDING MASTER DICTIONARY ---
Total tokens processed: 1,015,864
Calculating ALDF (Sketch Engine circular-distance logic)...
Dictionary saved as 'ALDF_Master_Dictionary.csv'.

--- PHASE 2: SCORING WORKSHOP ESSAYS ---
SUCCESS! Processed 10 essays.
Results saved to 'ALDF_Scores.csv'.
